In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sec_close = pd.read_csv('data/2026_01/sec_close.csv')

,issuer.name,issuer.tradingSymbol,reportingOwner.name,transactionDate,amounts.shares,amounts.pricePerShare,postTransactionAmounts.sharesOwnedFollowingTransaction,ownershipNature.directOrIndirectOwnership,reportingOwner.relationship.isDirector,reportingOwner.relationship.isOfficer,...,355,356,357,358,359,360,361,362,363,364
0,"REPUBLIC SERVICES, INC.",RSG,CASCADE INVESTMENT LLC,2009-01-28,409534.0000,26.4490,4.329457e+07,D,False,False,...,28.379999,28.590000,28.309999,28.200001,27.580000,27.580000,27.580000,27.430000,26.950001,27.040001
1,ACADIA REALTY TRUST,AKR,Spitz William T.,2009-01-30,258.0000,12.5800,6.176000e+03,D,True,False,...,17.410000,16.889999,16.469999,16.469999,16.469999,16.240000,15.970000,16.280001,16.059999,15.930000
2,ACADIA REALTY TRUST,AKR,CROCKER DOUGLAS II,2009-01-30,51.0000,12.5800,1.383000e+03,D,True,False,...,17.410000,16.889999,16.469999,16.469999,16.469999,16.240000,15.970000,16.280001,16.059999,15.930000
3,ACADIA REALTY TRUST,AKR,WIELANSKY LEE S,2009-01-30,886.0000,12.5800,2.116700e+04,D,True,False,...,17.410000,16.889999,16.469999,16.469999,16.469999,16.240000,15.970000,16.280001,16.059999,15.930000
4,ACADIA REALTY TRUST,AKR,KELLAR LORRENCE T,2009-01-30,356.0000,12.5800,8.499000e+03,D,True,False,...,17.410000,16.889999,16.469999,16.469999,16.469999,16.240000,15.970000,16.280001,16.059999,15.930000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12766,COHEN & STEERS TOTAL RETURN REALTY FUND INC,RFI,SCAPELL WILLIAM F,2023-08-18,0.6112,11.1600,0.000000e+00,I,False,True,...,12.280000,12.300000,12.350000,12.350000,12.350000,12.240000,12.320000,12.410000,12.430000,12.340000
12767,BARNWELL INDUSTRIES INC,BRN,SHERWOOD NED L,2023-08-31,81676.0000,2.5900,2.312929e+06,I,False,False,...,2.250000,2.270000,2.250000,2.320000,2.320000,2.320000,2.350000,2.330000,2.490000,2.440000
12768,TELOS CORP,TLS,Wood John B,2023-08-31,35000.0000,2.5128,5.048604e+06,D,True,True,...,3.620000,3.650000,3.330000,3.370000,3.370000,3.370000,3.180000,3.160000,3.700000,3.760000
12769,"ZIMMER BIOMET HOLDINGS, INC.",ZBH,Kolli Sreelakshmi,2023-08-30,1000.0000,120.3700,1.000000e+03,D,True,False,...,111.769997,111.639999,112.129997,113.419998,115.050003,115.050003,115.050003,114.629997,114.669998,114.190002


In [ ]:

df = sec_close.copy()

# change string into bool, D = direct, I = indirect
df['direct_ownership'] = (
    df['ownershipNature.directOrIndirectOwnership'].eq('D')
).astype('int8')

# for model readability month_sin and month_cos
df['transaction_month'] = pd.DatetimeIndex(
    df['transactionDate']).month
df["month_sin"] = np.sin(2 * np.pi * df["transaction_month"] / 12) 
df["month_cos"] = np.cos(2 * np.pi * df["transaction_month"] / 12)

"""
plt.plot(df["month_sin"], df["month_cos"])
plt.title("Transformation of transaction_month in month_sin and month_cos")
plt.xlabel("month_sin")
plt.ylabel("month_cos")
plt.show()

df = df.drop(columns="transaction_month")

"""

# count of fillings per person
# clean names because no id exported
df['reportingOwner.name'] = (
    df['reportingOwner.name']
    .str.replace(r'[^A-Za-z ]+', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.upper())

count_trades_tbl = (df
                    .groupby('reportingOwner.name')['issuer.tradingSymbol']
                    .count()
                    .rename('count')
                    .reset_index())
df = df.merge(
    right=count_trades_tbl.rename(
        columns={"count": "filing_count_reportingOwner.name"},
    ),
    on="reportingOwner.name",
    how="left",
)

# high frequency trader
median = df['filing_count_reportingOwner.name'].median()

df['high_frequency_trader'] = (
    df['filing_count_reportingOwner.name']
    .gt(median)
    .astype('int8')
)

# Cluster buys in past 14 days
df['transactionDate'] = pd.to_datetime(df['transactionDate'], errors='coerce')

# calculate daily fillings of ticker
daily = (df.groupby(['issuer.tradingSymbol', 'transactionDate']).size()
           .rename('n').reset_index())
# rolling count for past 14 days of fillings for this ticker
roll = (daily.set_index('transactionDate')
             .groupby('issuer.tradingSymbol')['n']
             .rolling('14D').sum()
             .rename('trades_14d')
             .reset_index())

df = df.merge(roll, on=['issuer.tradingSymbol', 'transactionDate'], how='left')

# Cluster buys dummy
df['cluster_buy'] = (\
    df['trades_14d'].gt(1)  # .gt() -> grater than
    .astype('int8')
)

# high price dummy
median = df['0'].median()

df['high_price'] = (
    df['0']
    .gt(median)
    .astype('int8')
)

# postTransactionAmounts.sharesOwnedFollowingTransaction is the amount of
# shares after each filling
post_shares = 'postTransactionAmounts.sharesOwnedFollowingTransaction'
shares = 'amounts.shares'

holdings_before_filing = (
    df[post_shares]
    - df[shares]
)

# calculation: (amount of shares in this filling/pre amount holdings)*100 for
# percent to know how much the person bought in comparison to what they owned
# pre amount of shares = post_shares - shares
# if old amount of shares = 0 then division by 0 would cause problems
df['holding_change_percent'] = np.where(
    holdings_before_filing == 0, 0, (df['amounts.shares'] /
                                     holdings_before_filing) * 100)
# remove implausible data
df = df[
    df['holding_change_percent'] >= 0]


# high change in holdings dummy
df['high_change_in_holdings'] = (
    df['holding_change_percent'] >
    df['holding_change_percent']
    .median()).astype('int8')

,issuer.name,issuer.tradingSymbol,reportingOwner.name,transactionDate,amounts.shares,amounts.pricePerShare,postTransactionAmounts.sharesOwnedFollowingTransaction,ownershipNature.directOrIndirectOwnership,reportingOwner.relationship.isDirector,reportingOwner.relationship.isOfficer,...,361,362,363,364,direct_ownership,transaction_month,month_sin,month_cos,filing_count_reportingOwner.name,high_frequency_trader
0,"REPUBLIC SERVICES, INC.",RSG,CASCADE INVESTMENT LLC,2009-01-28,409534.0000,26.4490,4.329457e+07,D,False,False,...,27.580000,27.430000,26.950001,27.040001,1,1,0.500000,0.866025,61,1
1,ACADIA REALTY TRUST,AKR,SPITZ WILLIAM T,2009-01-30,258.0000,12.5800,6.176000e+03,D,True,False,...,15.970000,16.280001,16.059999,15.930000,1,1,0.500000,0.866025,1,0
2,ACADIA REALTY TRUST,AKR,CROCKER DOUGLAS II,2009-01-30,51.0000,12.5800,1.383000e+03,D,True,False,...,15.970000,16.280001,16.059999,15.930000,1,1,0.500000,0.866025,1,0
3,ACADIA REALTY TRUST,AKR,WIELANSKY LEE S,2009-01-30,886.0000,12.5800,2.116700e+04,D,True,False,...,15.970000,16.280001,16.059999,15.930000,1,1,0.500000,0.866025,1,0
4,ACADIA REALTY TRUST,AKR,KELLAR LORRENCE T,2009-01-30,356.0000,12.5800,8.499000e+03,D,True,False,...,15.970000,16.280001,16.059999,15.930000,1,1,0.500000,0.866025,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12766,COHEN & STEERS TOTAL RETURN REALTY FUND INC,RFI,SCAPELL WILLIAM F,2023-08-18,0.6112,11.1600,0.000000e+00,I,False,True,...,12.320000,12.410000,12.430000,12.340000,0,8,-0.866025,-0.500000,4,0
12767,BARNWELL INDUSTRIES INC,BRN,SHERWOOD NED L,2023-08-31,81676.0000,2.5900,2.312929e+06,I,False,False,...,2.350000,2.330000,2.490000,2.440000,0,8,-0.866025,-0.500000,1,0
12768,TELOS CORP,TLS,WOOD JOHN B,2023-08-31,35000.0000,2.5128,5.048604e+06,D,True,True,...,3.180000,3.160000,3.700000,3.760000,1,8,-0.866025,-0.500000,9,0
12769,"ZIMMER BIOMET HOLDINGS, INC.",ZBH,KOLLI SREELAKSHMI,2023-08-30,1000.0000,120.3700,1.000000e+03,D,True,False,...,115.050003,114.629997,114.669998,114.190002,1,8,-0.866025,-0.500000,1,0


In [ ]:

# Target analysis
days = [4, 198]  # interesting spike after 4 days and maximum return on day 198
target_help_cols = [f'percent_change_since_{d}d' for d in days]

# calculate percent change from day 4 and 198 to filling date
for d in days:
    col = f'percent_change_since_{d}d'
    df[col] = (
        (df[str(d)] - df['0'])
        / df['0'] * 100
    )

# compute target variables
target_help_cols = [f'percent_change_since_{d}d' for d in days]

for p in range(1, 11):  # .gt() -> >=
    target = df[target_help_cols].ge(p).astype('int8')
    target.columns = [f't_{p}_{c}' for c in target_help_cols]
    df[target.columns] = target

# find most interesting target variables
target_cols = [f't_{p}_percent_change_since_{d}d'
               for d in days for p in range(1, 11)]

n = len(df)
ones = df[target_cols].sum().astype(int)
zeros = n - ones

summary = pd.DataFrame({
    'Target Name': target_cols,
    'ones':  ones.values,
    'zeros': zeros.values,
    'ones_rate':  (ones / n).values,
})

# keep only relevant targets
target_cols = [f't_{p}_percent_change_since_{d}d'
               for d in days for p in range(1, 11)]
target_cols.remove('t_1_percent_change_since_4d')
target_cols.remove('t_10_percent_change_since_198d')
df = df.drop(columns=target_cols)


In [ ]:


# Cluster buys dummy
df['cluster_buy'] = (
    df['trades_14d'].gt(1)  # .gt() -> grater than
    .astype('int8')
)

# high price dummy
median = df['0'].median()

df['high_price'] = (
    df['0']
    .gt(median)
    .astype('int8')
)
# postTransactionAmounts.sharesOwnedFollowingTransaction is the amount of
# shares after each filling
post_shares = 'postTransactionAmounts.sharesOwnedFollowingTransaction'
shares = 'amounts.shares'

holdings_before_filing = (
    df[post_shares]
    - df[shares]
)

# calculation: (amount of shares in this filling/pre amount holdings)*100 for
# percent to know how much the person bought in comparison to what they owned
# pre amount of shares = post_shares - shares
# if old amount of shares = 0 then division by 0 would cause problems
df['holding_change_percent'] = np.where(
    holdings_before_filing == 0, 0, (df['amounts.shares'] /
                                     holdings_before_filing) * 100)

# remove implausible data (132 cases)
df = df[
    df['holding_change_percent'] >= 0]

# high change in holdings dummy
df['high_change_in_holdings'] = (
    df['holding_change_percent'] >
    df['holding_change_percent']
    .median()).astype('int8')


# Target analysis
days = [4, 198]  # interesting spike after 4 days and maximum return on day 198
target_help_cols = [f'percent_change_since_{d}d' for d in days]

# calculate percent change from day 4 and 198 to filling date
for d in days:
    col = f'percent_change_since_{d}d'
    df[col] = (
        (df[str(d)] - df['0'])
        / df['0'] * 100
    )

# compute target variables
target_help_cols = [f'percent_change_since_{d}d' for d in days]

for p in range(1, 11):  # .gt() -> >=
    target = df[target_help_cols].ge(p).astype('int8')
    target.columns = [f't_{p}_{c}' for c in target_help_cols]
    df[target.columns] = target

# find most interesting target variables
target_cols = [f't_{p}_percent_change_since_{d}d'
               for d in days for p in range(1, 11)]

n = len(df)
ones = df[target_cols].sum().astype(int)
zeros = n - ones

summary = pd.DataFrame({
    'Target Name': target_cols,
    'ones':  ones.values,
    'zeros': zeros.values,
    'ones_rate':  (ones / n).values,
})

# keep only relevant targets
target_cols = [f't_{p}_percent_change_since_{d}d'
               for d in days for p in range(1, 11)]
target_cols.remove('t_1_percent_change_since_4d')
target_cols.remove('t_10_percent_change_since_198d')
df = df.drop(columns=target_cols)
